In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU List + Mapping Bootstrap, strictly 
      filtered by 'jurisdiction' for Canada, Hong Kong, and Barbados.
   2. STRING MAPPING: Maps engagements to AUs by checking if the AU_Name exists 
      inside the text of the 'owning_lob' or 'lob_using' columns.
   3. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double to find contracts >= 1,000,000.
   4. AGGREGATING (NUMERATOR & DENOMINATOR): Counts total DISTINCT engagements 
      (Denominator) and total DISTINCT engagements >= 1MM (Numerator).
   5. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(business_segment) AS Business_Segment 
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has cost centers mapped to it
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Third_Party_Risk_Data AS (
    -- 4. Extract TP data, clean the contract amount string, and cast to double
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Engagements_By_AU AS (
    -- 5. Map engagements to AUs via string matching and calculate Num/Denom
    SELECT 
        a.Base_AU_ID,
        -- Total Distinct Engagements
        COUNT(DISTINCT tp.EngagementNumber) AS Denominator,
        -- Distinct Engagements >= 1 Million
        COUNT(DISTINCT CASE WHEN tp.Cleaned_Amount >= 1000000 THEN tp.EngagementNumber END) AS Numerator
    FROM All_Base_AUs a
    -- Joins if the AU Name is found anywhere inside the owning_lob or lob_using strings
    INNER JOIN Third_Party_Risk_Data tp
        ON UPPER(tp.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
        OR UPPER(tp.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    GROUP BY a.Base_AU_ID
)

-- 6. Final Template: Strict 6-column output with Percentage Math
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Base_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - TP High Value Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.abac_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   PURPOSE: 
   Provides a row-by-row view of every Third Party Engagement mapped to an AU.
   Displays the raw Contract Amount string, the cleaned numeric value, and strictly 
   flags if it counts toward the Numerator (>= 1,000,000) or just the Denominator.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND UPPER(TRIM(jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        Contract_Amount AS Raw_Contract_Amount,
        Currency_code,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    tp.EngagementNumber,
    tp.ThirdPartyName,
    tp.owning_lob AS Raw_Col_K_owning_lob,
    tp.lob_using AS Raw_Col_L_lob_using,
    tp.Raw_Contract_Amount,
    tp.Currency_code,
    tp.Cleaned_Amount,
    
    CASE 
        WHEN tp.Cleaned_Amount >= 1000000 THEN 'Yes (Added to Numerator)'
        ELSE 'No (Denominator Only)' 
    END AS Is_High_Value_Contract
    
FROM All_Base_AUs a

-- Inner join via string matching
INNER JOIN Third_Party_Risk_Data tp
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(a.AU_Name) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(a.AU_Name) || '%'
    
ORDER BY a.Base_AU_ID;